# Five-threshold interpretation and structural robustness

This notebook tests whether the five-scale conclusions survive predeclared changes in symbol, absolute scale and threshold-grid geometry.

The primary hierarchy is a constant ratio of 1.5. Six cases are fixed before their results are inspected:

- GBPUSD at 15 bps, 20 bps and 25 bps starting thresholds;
- EURUSD, GBPUSD and USDJPY at the 20 bps starting threshold;
- GBPUSD with the original linear 20/40/60/80/100 bps grid as a sensitivity case.

All cross-symbol cases use one common calendar interval. No case is retuned after observing another case.

The notebook evaluates structural recurrence, not trading profitability. Terminal outcomes and outcome-conditioned AUC remain diagnostic hindsight. Every checkpoint feature itself is causal and sampled strictly before stage resolution.

In [1]:
from enum import StrEnum
from pathlib import Path
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dc_events

from dc_basics.data import resolve_data_root

DATA_ROOT = resolve_data_root()
CHECKPOINT_BARS = [0, 20, 80, 160, 320]
MIN_AUC_CLASS_COUNT = 20
MIN_OUTER_EVENTS = 75
MIN_STRICT_LINEAGES = 100

In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"
    OUTER = "outer"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class Stage(StrEnum):
    SMALL_TO_MEDIUM = "small_to_medium"
    MEDIUM_TO_LARGE = "medium_to_large"
    LARGE_TO_MACRO = "large_to_macro"
    MACRO_TO_OUTER = "macro_to_outer"


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    STOPPED_BELOW_MACRO = "stopped_below_macro"
    STOPPED_BELOW_OUTER = "stopped_below_outer"
    REACHED_OUTER = "reached_outer"
    AMBIGUOUS_STAGE = "ambiguous_stage"
    UNRESOLVED_STAGE = "unresolved_stage"
    UNLINKED_HANDOFF = "unlinked_handoff"


class GridFamily(StrEnum):
    CONSTANT_RATIO = "constant_ratio"
    LINEAR = "linear"


class ComparisonAxis(StrEnum):
    BASELINE = "baseline"
    CROSS_SYMBOL = "cross_symbol"
    ABSOLUTE_SCALE = "absolute_scale"
    GRID_SENSITIVITY = "grid_sensitivity"


class CaseId(StrEnum):
    GBP_RATIO_15 = "GBPUSD ratio-1.5 start-15bp"
    GBP_RATIO_20 = "GBPUSD ratio-1.5 start-20bp"
    GBP_RATIO_25 = "GBPUSD ratio-1.5 start-25bp"
    EUR_RATIO_20 = "EURUSD ratio-1.5 start-20bp"
    JPY_RATIO_20 = "USDJPY ratio-1.5 start-20bp"
    GBP_LINEAR_20 = "GBPUSD linear 20/40/60/80/100bp"


class CaseSpec(NamedTuple):
    case_id: CaseId
    symbol: str
    thresholds: tuple[float, float, float, float, float]
    grid_family: GridFamily
    comparison_axis: ComparisonAxis


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO, Scale.OUTER)
SCALE_PREFIXES = {
    Scale.SMALL: "S",
    Scale.MEDIUM: "M",
    Scale.LARGE: "L",
    Scale.MACRO: "X",
    Scale.OUTER: "O",
}
STAGE_ORDER = (
    Stage.SMALL_TO_MEDIUM,
    Stage.MEDIUM_TO_LARGE,
    Stage.LARGE_TO_MACRO,
    Stage.MACRO_TO_OUTER,
)
STAGE_SPECS = {
    Stage.SMALL_TO_MEDIUM: (Scale.SMALL, Scale.MEDIUM),
    Stage.MEDIUM_TO_LARGE: (Scale.MEDIUM, Scale.LARGE),
    Stage.LARGE_TO_MACRO: (Scale.LARGE, Scale.MACRO),
    Stage.MACRO_TO_OUTER: (Scale.MACRO, Scale.OUTER),
}
STOPPED_OUTCOMES = {
    0: TerminalOutcome.STOPPED_BELOW_MEDIUM,
    1: TerminalOutcome.STOPPED_BELOW_LARGE,
    2: TerminalOutcome.STOPPED_BELOW_MACRO,
    3: TerminalOutcome.STOPPED_BELOW_OUTER,
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}


def ratio_ladder(first_threshold: float, ratio: float = 1.5) -> tuple[float, ...]:
    return tuple(first_threshold * ratio ** power for power in range(5))


CASE_SPECS = (
    CaseSpec(
        CaseId.GBP_RATIO_15,
        "GBPUSD.r",
        ratio_ladder(0.0015),
        GridFamily.CONSTANT_RATIO,
        ComparisonAxis.ABSOLUTE_SCALE,
    ),
    CaseSpec(
        CaseId.GBP_RATIO_20,
        "GBPUSD.r",
        ratio_ladder(0.0020),
        GridFamily.CONSTANT_RATIO,
        ComparisonAxis.BASELINE,
    ),
    CaseSpec(
        CaseId.GBP_RATIO_25,
        "GBPUSD.r",
        ratio_ladder(0.0025),
        GridFamily.CONSTANT_RATIO,
        ComparisonAxis.ABSOLUTE_SCALE,
    ),
    CaseSpec(
        CaseId.EUR_RATIO_20,
        "EURUSD.r",
        ratio_ladder(0.0020),
        GridFamily.CONSTANT_RATIO,
        ComparisonAxis.CROSS_SYMBOL,
    ),
    CaseSpec(
        CaseId.JPY_RATIO_20,
        "USDJPY.r",
        ratio_ladder(0.0020),
        GridFamily.CONSTANT_RATIO,
        ComparisonAxis.CROSS_SYMBOL,
    ),
    CaseSpec(
        CaseId.GBP_LINEAR_20,
        "GBPUSD.r",
        (0.0020, 0.0040, 0.0060, 0.0080, 0.0100),
        GridFamily.LINEAR,
        ComparisonAxis.GRID_SENSITIVITY,
    ),
)

## Predeclared comparison contract

Three comparisons answer different questions:

1. Cross-symbol robustness holds the constant-ratio 20 bps ladder fixed and changes GBPUSD, EURUSD and USDJPY.
2. Absolute-scale robustness holds GBPUSD and the 1.5 ratio fixed while changing the first threshold to 15, 20 and 25 bps.
3. Grid sensitivity holds GBPUSD and approximately the same outer scale fixed while comparing constant-ratio and linear spacing.

A structural conclusion is credible when its qualitative direction and economic interpretation recur with adequate samples. Exact probabilities are expected to differ. Failure in one case is reported, not optimized away.

In [3]:
case_design = pd.DataFrame.from_records([
    {
        "case": spec.case_id.value,
        "symbol": spec.symbol,
        "grid_family": spec.grid_family.value,
        "comparison_axis": spec.comparison_axis.value,
        **{
            f"{scale.value}_threshold": threshold
            for scale, threshold in zip(SCALE_ORDER, spec.thresholds, strict=True)
        },
        **{
            f"ratio_{lower.value}_{upper.value}": upper_threshold / lower_threshold
            for lower, upper, lower_threshold, upper_threshold in zip(
                SCALE_ORDER[:-1],
                SCALE_ORDER[1:],
                spec.thresholds[:-1],
                spec.thresholds[1:],
                strict=True,
            )
        },
    }
    for spec in CASE_SPECS
]).set_index("case")

threshold_columns = [f"{scale.value}_threshold" for scale in SCALE_ORDER]
ratio_columns = [
    f"ratio_{lower.value}_{upper.value}"
    for lower, upper in zip(SCALE_ORDER[:-1], SCALE_ORDER[1:], strict=True)
]
display(case_design.style.format({
    **{column: "{:.4%}" for column in threshold_columns},
    **{column: "{:.3f}" for column in ratio_columns},
}))

,symbol,grid_family,comparison_axis,small_threshold,medium_threshold,large_threshold,macro_threshold,outer_threshold,ratio_small_medium,ratio_medium_large,ratio_large_macro,ratio_macro_outer
case,,,,,,,,,,,,
GBPUSD ratio-1.5 start-15bp,GBPUSD.r,constant_ratio,absolute_scale,0.1500%,0.2250%,0.3375%,0.5062%,0.7594%,1.500,1.500,1.500,1.500
GBPUSD ratio-1.5 start-20bp,GBPUSD.r,constant_ratio,baseline,0.2000%,0.3000%,0.4500%,0.6750%,1.0125%,1.500,1.500,1.500,1.500
GBPUSD ratio-1.5 start-25bp,GBPUSD.r,constant_ratio,absolute_scale,0.2500%,0.3750%,0.5625%,0.8438%,1.2656%,1.500,1.500,1.500,1.500
EURUSD ratio-1.5 start-20bp,EURUSD.r,constant_ratio,cross_symbol,0.2000%,0.3000%,0.4500%,0.6750%,1.0125%,1.500,1.500,1.500,1.500
USDJPY ratio-1.5 start-20bp,USDJPY.r,constant_ratio,cross_symbol,0.2000%,0.3000%,0.4500%,0.6750%,1.0125%,1.500,1.500,1.500,1.500
GBPUSD linear 20/40/60/80/100bp,GBPUSD.r,linear,grid_sensitivity,0.2000%,0.4000%,0.6000%,0.8000%,1.0000%,2.000,1.500,1.333,1.250


## Load a common cross-symbol interval

Restricting every symbol to the same start and end dates prevents differing sample windows from masquerading as symbol effects.

In [4]:
from dc_basics.data import load_close as _load_close

def load_close(symbol: str) -> pd.Series:
    return _load_close(DATA_ROOT, symbol)


symbols = sorted({spec.symbol for spec in CASE_SPECS})
unrestricted_prices = {symbol: load_close(symbol) for symbol in symbols}
common_start = max(close.index[0] for close in unrestricted_prices.values())
common_end = min(close.index[-1] for close in unrestricted_prices.values())
if common_start >= common_end:
    raise ValueError("The symbols have no common date interval.")

prices_by_symbol = {
    symbol: close.loc[common_start:common_end]
    for symbol, close in unrestricted_prices.items()
}
common_interval_summary = pd.DataFrame.from_records([
    {
        "symbol": symbol,
        "bars": len(close),
        "start": close.index[0],
        "end": close.index[-1],
    }
    for symbol, close in prices_by_symbol.items()
]).set_index("symbol")
print(f"Common interval: {common_start} to {common_end}")
common_interval_summary

Common interval: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


,bars,start,end
symbol,,,
EURUSD.r,258848,2023-05-22 14:00:00+02:00,2026-11-09 13:30:00+02:00
GBPUSD.r,258573,2023-05-22 14:00:00+02:00,2026-11-09 13:30:00+02:00
USDJPY.r,258667,2023-05-22 14:00:00+02:00,2026-11-09 13:30:00+02:00


## Generic event, contest and linkage machinery

Every case is reconstructed independently. Event IDs are local to a case and linked only by exact equality inside that case.

In [5]:
from dc_basics.events import prepare_event_ledger as _prepare_event_ledger

def prepare_event_ledger(source: pd.DataFrame, scale: Scale, theta: float) -> pd.DataFrame:
    return _prepare_event_ledger(source, SCALE_PREFIXES[scale], scale, theta)


from dc_basics.hierarchy import build_adjacent_attempts


In [6]:
from dc_basics.lineage import build_five_scale_linkages


## Compact causal checkpoint extraction

Only fixed pre-resolution checkpoints are retained. This avoids storing six full bar-by-bar live panels while preserving the causal comparisons required for robustness.

In [7]:
def classify_modes(modes: tuple[str, ...]) -> dict[str, object]:
    boundary_count = sum(left != right for left, right in zip(modes[:-1], modes[1:], strict=True))
    outer_agreement_breadth = sum(mode == modes[-1] for mode in modes)
    small_coherence_depth = 1
    for mode in modes[1:]:
        if mode != modes[0]:
            break
        small_coherence_depth += 1
    if boundary_count == 0:
        shape = "broad_agreement"
    elif boundary_count == 1:
        shape = f"counter_depth_{small_coherence_depth}"
    else:
        shape = f"fragmented_{boundary_count}_boundaries"
    return {
        "configuration": "/".join(modes),
        "hierarchy_shape": shape,
        "direction_boundaries": boundary_count,
        "outer_agreement_breadth": outer_agreement_breadth,
        "coherent_frontier": boundary_count <= 1,
    }


from dc_basics.live import build_causal_running_extreme


def target_progress_at(
    price: float, extreme: float, target_mode: str, theta: float
) -> float:
    if target_mode == DcMode.UP.value:
        return (price - extreme) / extreme / theta
    if target_mode == DcMode.DOWN.value:
        return (extreme - price) / extreme / theta
    raise ValueError(f"Unsupported target mode: {target_mode}")

In [8]:
def build_checkpoint_rows(
    case_name: str,
    close: pd.Series,
    mode_frame: pd.DataFrame,
    live_frames: dict[Scale, pd.DataFrame],
    running_extremes: dict[Scale, pd.Series],
    dcc_cumulative: dict[Scale, np.ndarray],
    strict_lineages: pd.DataFrame,
    strict_stages: pd.DataFrame,
    thresholds: dict[Scale, float],
) -> pd.DataFrame:
    lineage_lookup = strict_lineages.set_index("lineage_id", drop=False)
    records = []

    for stage_row in strict_stages.itertuples(index=False):
        stage = Stage(stage_row.stage)
        lower_scale, upper_scale = STAGE_SPECS[stage]
        lineage = lineage_lookup.loc[stage_row.lineage_id]
        start_position = int(stage_row.start_position)
        resolution_position = int(stage_row.resolution_position)

        for elapsed_bars in CHECKPOINT_BARS:
            position = start_position + elapsed_bars
            if position >= resolution_position:
                continue

            modes = tuple(
                mode_frame.iloc[position][f"{scale.value}_mode"]
                for scale in SCALE_ORDER
            )
            if any(pd.isna(mode) for mode in modes):
                raise AssertionError("A checkpoint has an uninitialized mode.")
            properties = classify_modes(modes)
            price = float(close.iloc[position])
            upper_extreme = float(running_extremes[upper_scale].iloc[position])
            progress = target_progress_at(
                price,
                upper_extreme,
                stage_row.target_mode,
                thresholds[upper_scale],
            )
            if progress >= 1.0 + 1e-10:
                raise AssertionError("A pre-resolution checkpoint crossed its upper threshold.")

            lower_live = live_frames[lower_scale].iloc[position]
            upper_live = live_frames[upper_scale].iloc[position]
            # Lower-scale DCCs inside a stage are zero by construction: the first one
            # resolves the stage. Small-clock activity is informative from Stage 2 on.
            nested_small_dccs = (
                dcc_cumulative[Scale.SMALL][position]
                - dcc_cumulative[Scale.SMALL][start_position]
            )
            records.append({
                "case": case_name,
                "lineage_id": stage_row.lineage_id,
                "stage": stage.value,
                "elapsed_bars": elapsed_bars,
                "time": close.index[position],
                "bar_position": position,
                "target_mode": stage_row.target_mode,
                "target_progress": progress,
                "lower_tmv_live": lower_live["tmv_current"],
                "lower_t_live": lower_live["t_current"],
                "lower_r_live": lower_live["rdc_current"],
                "upper_tmv_live": upper_live["tmv_current"],
                "upper_t_live": upper_live["t_current"],
                "upper_r_live": upper_live["rdc_current"],
                "nested_small_dccs": nested_small_dccs,
                "nested_small_dcc_rate": nested_small_dccs / max(elapsed_bars, 1),
                **properties,
                "future_stage_outcome": stage_row.outcome,
                "stage_outcome_information_time": stage_row.resolution_time,
                "future_terminal_depth": int(lineage["terminal_depth"]),
                "terminal_outcome_information_time": lineage["terminal_time"],
            })

    checkpoints = pd.DataFrame.from_records(records)
    if not checkpoints["time"].lt(checkpoints["stage_outcome_information_time"]).all():
        raise AssertionError("A checkpoint includes its stage-resolution bar.")
    if not checkpoints["time"].lt(checkpoints["terminal_outcome_information_time"]).all():
        raise AssertionError("A checkpoint includes known terminal outcome.")
    return checkpoints

In [9]:
def build_case_structure(close: pd.Series, spec: CaseSpec) -> dict[str, object]:
    thresholds = dict(zip(SCALE_ORDER, spec.thresholds, strict=True))
    event_ledgers = {}
    live_frames = {}
    mode_frame = pd.DataFrame(index=close.index)

    for scale in SCALE_ORDER:
        theta = thresholds[scale]
        ledger = prepare_event_ledger(parse_dc_events(prices=close, theta=theta), scale, theta)
        live = dc_live_features(close, theta=theta)
        event_ledgers[scale] = ledger
        live_frames[scale] = live
        mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
        mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
        mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
        mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(
            ledger.set_index("dcc_idx")["event_id"]
        )

    mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
    known_modes = mode_frame.loc[mode_frame[mode_columns].notna().all(axis=1), mode_columns]
    mode_matrix = known_modes.to_numpy()
    direction_boundaries = np.sum(mode_matrix[:, :-1] != mode_matrix[:, 1:], axis=1)
    outer_agreement_breadth = np.sum(mode_matrix == mode_matrix[:, [-1]], axis=1)
    configurations = pd.Series(
        ["/".join(row) for row in mode_matrix],
        index=known_modes.index,
    )
    probabilities = configurations.value_counts(normalize=True)
    entropy_bits = -(probabilities * np.log2(probabilities)).sum()
    state_summary = {
        "initialized_bars": len(known_modes),
        "observed_configurations": configurations.nunique(),
        "configuration_entropy_bits": entropy_bits,
        "effective_configurations": 2.0 ** entropy_bits,
        "broad_agreement_share": np.mean(direction_boundaries == 0),
        "coherent_frontier_share": np.mean(direction_boundaries <= 1),
        "fragmented_share": np.mean(direction_boundaries >= 2),
        "mean_direction_boundaries": np.mean(direction_boundaries),
        "mean_outer_agreement_breadth": np.mean(outer_agreement_breadth),
    }

    attempts_by_stage = {}
    for stage in STAGE_ORDER:
        lower_scale, upper_scale = STAGE_SPECS[stage]
        attempts = build_adjacent_attempts(mode_frame, lower_scale, upper_scale)
        attempts.insert(0, "stage", stage.value)
        attempts_by_stage[stage] = attempts

    previous_modes = mode_frame[mode_columns].shift(1)
    small_attempts = attempts_by_stage[Stage.SMALL_TO_MEDIUM]
    seed_times = pd.DatetimeIndex(small_attempts["start_time"])
    seed_previous_modes = previous_modes.loc[seed_times]
    eligible = (
        seed_previous_modes.notna().all(axis=1)
        & seed_previous_modes.nunique(axis=1).eq(1)
    ).to_numpy()
    seeds = small_attempts.loc[eligible].copy()
    linkages, stages = build_five_scale_linkages(seeds, attempts_by_stage)
    resolved_lineages = linkages.loc[linkages["resolved_lineage"]].copy()
    strict_lineages = linkages.loc[linkages["strict_clean"]].copy()
    terminal_outcomes = linkages["terminal_outcome"]
    lineage_accounting = {
        "case": spec.case_id.value,
        "eligible_roots": len(linkages),
        "resolved_lineages": len(resolved_lineages),
        "strict_clean_lineages": len(strict_lineages),
        "resolved_with_same_bar_stage": int(
            (linkages["resolved_lineage"] & linkages["same_bar_stage_count"].gt(0)).sum()
        ),
        "ambiguous_stage": int(terminal_outcomes.eq(TerminalOutcome.AMBIGUOUS_STAGE.value).sum()),
        "unresolved_stage": int(terminal_outcomes.eq(TerminalOutcome.UNRESOLVED_STAGE.value).sum()),
        "unlinked_handoff": int(terminal_outcomes.eq(TerminalOutcome.UNLINKED_HANDOFF.value).sum()),
        "same_bar_stages": int(stages["resolution_bars"].eq(0).sum()),
    }
    if lineage_accounting["eligible_roots"] != (
        lineage_accounting["strict_clean_lineages"]
        + lineage_accounting["resolved_with_same_bar_stage"]
        + lineage_accounting["ambiguous_stage"]
        + lineage_accounting["unresolved_stage"]
        + lineage_accounting["unlinked_handoff"]
    ):
        raise AssertionError(f"{spec.case_id.value}: lineage accounting does not reconcile.")
    strict_ids = set(strict_lineages["lineage_id"])
    strict_stages = stages.loc[stages["lineage_id"].isin(strict_ids)].copy()

    if not strict_stages["resolution_bars"].gt(0).all():
        raise AssertionError("A strict-clean stage has no pre-resolution bars.")
    # Rates (Audit 5 strata) use every resolved lineage, like the Audit 2 ladder; strict-clean
    # is reserved for live intervals.
    resolved_ids = set(resolved_lineages["lineage_id"])
    resolved_stages = stages.loc[stages["lineage_id"].isin(resolved_ids)].copy()

    # Calendar thirds of the common window, so every case splits on the same dates.
    strict_lineages["sample_third"] = pd.cut(
        strict_lineages["seed_time"],
        bins=pd.date_range(close.index[0], close.index[-1], periods=4),
        labels=["early", "middle", "late"],
        include_lowest=True,
    )
    sample_third_lookup = strict_lineages.set_index("lineage_id")["sample_third"]
    strict_stages["sample_third"] = strict_stages["lineage_id"].map(sample_third_lookup)
    resolved_thirds = pd.cut(
        resolved_lineages.set_index("lineage_id")["seed_time"],
        bins=pd.date_range(close.index[0], close.index[-1], periods=4),
        labels=["early", "middle", "late"],
        include_lowest=True,
    )
    resolved_stages["sample_third"] = resolved_stages["lineage_id"].map(resolved_thirds)

    running_extremes = {
        scale: build_causal_running_extreme(close, live_frames[scale])
        for scale in SCALE_ORDER
    }
    dcc_cumulative = {}
    for scale in SCALE_ORDER:
        flags = (
            mode_frame[f"{scale.value}_upward_dcc"]
            | mode_frame[f"{scale.value}_downward_dcc"]
        ).astype(np.int64)
        dcc_cumulative[scale] = flags.cumsum().to_numpy(dtype=np.int64)

    checkpoints = build_checkpoint_rows(
        spec.case_id.value,
        close,
        mode_frame,
        live_frames,
        running_extremes,
        dcc_cumulative,
        strict_lineages,
        strict_stages,
        thresholds,
    )

    clock_summary = pd.DataFrame.from_records([
        {
            "case": spec.case_id.value,
            "scale": scale.value,
            "threshold": thresholds[scale],
            "events": len(event_ledgers[scale]),
            "median_t": event_ledgers[scale]["t"].median(),
            "median_r": event_ledgers[scale]["r"].median(),
        }
        for scale in SCALE_ORDER
    ])
    return {
        "spec": spec,
        "clock_summary": clock_summary,
        "state_summary": state_summary,
        "lineage_accounting": lineage_accounting,
        "resolved_lineages": resolved_lineages,
        "strict_lineages": strict_lineages,
        "strict_stages": strict_stages,
        "resolved_stages": resolved_stages,
        "checkpoints": checkpoints,
    }

## Run the six predeclared cases

This is the expensive cell. Each case independently reconstructs five DC clocks, four adjacent contests, exact linkage and compact causal checkpoints. No result from one case changes another case's thresholds or rules.

In [10]:
case_results = {}
for spec in CASE_SPECS:
    print(f"Building {spec.case_id.value}...")
    case_results[spec.case_id] = build_case_structure(prices_by_symbol[spec.symbol], spec)

print(f"Completed {len(case_results)} predeclared cases.")

Building GBPUSD ratio-1.5 start-15bp...


Building GBPUSD ratio-1.5 start-20bp...


Building GBPUSD ratio-1.5 start-25bp...


Building EURUSD ratio-1.5 start-20bp...


Building USDJPY ratio-1.5 start-20bp...


Building GBPUSD linear 20/40/60/80/100bp...


Completed 6 predeclared cases.


## Audit 1: event-clock and state-space adequacy

A five-scale interpretation is not credible when the outer clock or strict lineage cohort is too sparse. Adequacy is reported explicitly rather than repaired by retuning.

The lineage accounting table comes first. Every eligible broad-agreement root is assigned to exactly one row category, and the categories reconcile to the root count:

- `strict_clean_lineages`: resolved, with every stage having at least one pre-resolution bar;
- `resolved_with_same_bar_stage`: resolved, but some stage resolved on its start bar. Such a stage can only be a propagation, so each of these lineages reached at least medium; a later stage can still have rejected it;
- `ambiguous_stage`, `unresolved_stage` (right-censored) and `unlinked_handoff`: no terminal depth.

`same_bar_stages` counts zero-length stages across all lineages.

In [11]:
clock_summary = pd.concat(
    [result["clock_summary"] for result in case_results.values()],
    ignore_index=True,
)
clock_pivot = clock_summary.pivot(index="case", columns="scale", values="events").reindex(
    columns=[scale.value for scale in SCALE_ORDER]
)

state_records = []
for case_id, result in case_results.items():
    strict_lineages = result["strict_lineages"]
    state_records.append({
        "case": case_id.value,
        **result["state_summary"],
        "strict_clean_lineages": len(strict_lineages),
        "outer_events": int(
            clock_summary.loc[
                clock_summary["case"].eq(case_id.value)
                & clock_summary["scale"].eq(Scale.OUTER.value),
                "events",
            ].iloc[0]
        ),
    })

state_summary = pd.DataFrame.from_records(state_records).set_index("case")
state_summary["adequate_outer_clock"] = state_summary["outer_events"].ge(MIN_OUTER_EVENTS)
state_summary["adequate_lineage_cohort"] = state_summary["strict_clean_lineages"].ge(
    MIN_STRICT_LINEAGES
)

lineage_accounting = pd.DataFrame.from_records([
    result["lineage_accounting"] for result in case_results.values()
]).set_index("case")
display(lineage_accounting.style.format("{:,}"))
display(clock_pivot.style.format("{:,}"))
display(state_summary.style.format({
    "initialized_bars": "{:,}",
    "observed_configurations": "{:,}",
    "configuration_entropy_bits": "{:.3f}",
    "effective_configurations": "{:.2f}",
    "broad_agreement_share": "{:.2%}",
    "coherent_frontier_share": "{:.2%}",
    "fragmented_share": "{:.2%}",
    "mean_direction_boundaries": "{:.3f}",
    "mean_outer_agreement_breadth": "{:.3f}",
    "strict_clean_lineages": "{:,}",
    "outer_events": "{:,}",
}))

,eligible_roots,resolved_lineages,strict_clean_lineages,resolved_with_same_bar_stage,ambiguous_stage,unresolved_stage,unlinked_handoff,same_bar_stages
case,,,,,,,,
GBPUSD ratio-1.5 start-15bp,"1,317","1,316","1,220",96,0,1,0,127
GBPUSD ratio-1.5 start-20bp,799,798,751,47,0,1,0,68
GBPUSD ratio-1.5 start-25bp,535,534,508,26,0,1,0,39
EURUSD ratio-1.5 start-20bp,700,699,647,52,0,1,0,68
USDJPY ratio-1.5 start-20bp,"1,223","1,222","1,070",152,0,1,0,218
GBPUSD linear 20/40/60/80/100bp,833,832,800,32,0,1,0,43


scale,small,medium,large,macro,outer
case,,,,,
EURUSD ratio-1.5 start-20bp,"2,419","1,305",625,298,144
GBPUSD linear 20/40/60/80/100bp,"2,697",839,392,220,146
GBPUSD ratio-1.5 start-15bp,"4,410","2,187","1,109",535,254
GBPUSD ratio-1.5 start-20bp,"2,697","1,389",683,308,146
GBPUSD ratio-1.5 start-25bp,"1,881",939,437,190,101
USDJPY ratio-1.5 start-20bp,"3,809","1,951","1,010",528,252


,initialized_bars,observed_configurations,configuration_entropy_bits,effective_configurations,broad_agreement_share,coherent_frontier_share,fragmented_share,mean_direction_boundaries,mean_outer_agreement_breadth,strict_clean_lineages,outer_events,adequate_outer_clock,adequate_lineage_cohort
case,,,,,,,,,,,,,
GBPUSD ratio-1.5 start-15bp,"258,007",32,4.370,20.68,26.34%,70.82%,29.18%,1.091,3.347,"1,220",254,True,True
GBPUSD ratio-1.5 start-20bp,"257,813",32,4.372,20.71,26.39%,70.63%,29.37%,1.095,3.369,751,146,True,True
GBPUSD ratio-1.5 start-25bp,"257,639",32,4.313,19.87,26.85%,72.82%,27.18%,1.059,3.353,508,101,True,True
EURUSD ratio-1.5 start-20bp,"258,030",32,4.423,21.45,24.63%,69.00%,31.00%,1.132,3.322,647,144,True,True
USDJPY ratio-1.5 start-20bp,"257,979",32,4.139,17.62,32.65%,72.21%,27.79%,1.010,3.525,"1,070",252,True,True
GBPUSD linear 20/40/60/80/100bp,"257,819",32,4.160,17.88,28.47%,73.51%,26.49%,1.029,3.635,800,146,True,True


In [12]:
state_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Broad-agreement share", "Fragmented share"],
)
state_figure.add_trace(go.Bar(
    x=state_summary.index,
    y=state_summary["broad_agreement_share"],
    marker_color="#2563eb",
    name="Broad agreement",
), row=1, col=1)
state_figure.add_trace(go.Bar(
    x=state_summary.index,
    y=state_summary["fragmented_share"],
    marker_color="#f97316",
    name="Fragmented",
), row=1, col=2)
state_figure.update_layout(
    title="Five-scale state occupancy across predeclared cases",
    template="plotly_white",
    height=550,
    showlegend=False,
)
state_figure.update_yaxes(tickformat=".0%", row=1, col=1)
state_figure.update_yaxes(tickformat=".0%", row=1, col=2)
state_figure.update_xaxes(tickangle=-25)
state_figure.show()

## Audit 2: linked terminal depth and cumulative reach

Terminal depth is hindsight. The comparison asks whether a recognizable propagation ladder recurs; exact depth probabilities are allowed to vary.

Two cohorts are shown. The **all-resolved** ladder keeps every lineage with a terminal depth. The **strict-clean** ladder also drops lineages with a same-bar stage. Every dropped lineage reached at least medium, so P(reach medium) is biased down by construction; later conditional rates are usually lower too, but not always, because a dropped lineage can be rejected at a later stage (compare the macro->outer rates across cases). Strict-clean is the right cohort where a non-empty live interval is required (Audits 3–4); the all-resolved ladder is the better description of linked depth.

In [13]:
depth_records = []
for case_id, result in case_results.items():
    for cohort, lineages in (
        ("all_resolved", result["resolved_lineages"]),
        ("strict_clean", result["strict_lineages"]),
    ):
        for depth in range(5):
            depth_records.append({
                "case": case_id.value,
                "cohort": cohort,
                "lineages": len(lineages),
                "depth": depth,
                "terminal_depth_share": lineages["terminal_depth"].astype(int).eq(depth).mean(),
                "cumulative_reach_rate": lineages["terminal_depth"].astype(int).ge(depth).mean(),
                "lineages_reaching_depth": lineages["terminal_depth"].astype(int).ge(depth).sum(),
            })

depth_summary = pd.DataFrame.from_records(depth_records)
display(depth_summary.pivot(
    index=["case", "cohort"],
    columns="depth",
    values="terminal_depth_share",
).style.format("{:.2%}"))
display(depth_summary.pivot(
    index=["case", "cohort"],
    columns="depth",
    values="cumulative_reach_rate",
).style.format("{:.2%}"))

reach_figure = go.Figure()
for (case, cohort), group in depth_summary.groupby(["case", "cohort"], sort=False):
    reach_figure.add_trace(go.Scatter(
        x=group["depth"],
        y=group["cumulative_reach_rate"],
        mode="lines+markers",
        name=f"{case} ({cohort})",
        line={"dash": "solid" if cohort == "all_resolved" else "dot"},
    ))
reach_figure.update_layout(
    title="Cumulative linked reach across five-scale cases (solid: all resolved; dotted: strict-clean)",
    template="plotly_white",
    height=600,
    xaxis_title="Attained depth",
    yaxis_title="Cumulative reach rate",
    yaxis_tickformat=".0%",
    legend={"orientation": "h", "y": 1.22, "x": 0.0},
)
reach_figure.show()

## Audit 3: survival conditioning

Later checkpoints contain only unresolved survivors. The later-propagation rate at a checkpoint describes that surviving cohort; it is not the causal effect of waiting.

In [14]:
checkpoint_rows = pd.concat(
    [result["checkpoints"] for result in case_results.values()],
    ignore_index=True,
)
stage_attempt_counts = checkpoint_rows.loc[checkpoint_rows["elapsed_bars"].eq(0)].groupby(
    ["case", "stage"]
)["lineage_id"].nunique()

survival_summary = checkpoint_rows.groupby(["case", "stage", "elapsed_bars"], sort=True).agg(
    surviving_attempts=("lineage_id", "nunique"),
    later_propagation_rate=(
        "future_stage_outcome",
        lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean(),
    ),
    target_progress_median=("target_progress", "median"),
    lower_r_median=("lower_r_live", "median"),
    direction_boundaries_median=("direction_boundaries", "median"),
    outer_agreement_breadth_median=("outer_agreement_breadth", "median"),
    nested_small_dccs_median=("nested_small_dccs", "median"),
).reset_index()
survival_summary["survivor_share"] = survival_summary.apply(
    lambda row: row["surviving_attempts"] / stage_attempt_counts.loc[(row["case"], row["stage"])],
    axis=1,
)

display(survival_summary.set_index(["case", "stage", "elapsed_bars"]).style.format({
    "surviving_attempts": "{:,}",
    "later_propagation_rate": "{:.2%}",
    "target_progress_median": "{:.2%}",
    "lower_r_median": "{:.3e}",
    "direction_boundaries_median": "{:.1f}",
    "outer_agreement_breadth_median": "{:.1f}",
    "nested_small_dccs_median": "{:.1f}",
    "survivor_share": "{:.2%}",
}))

## Audit 4: live-feature overlap

Orientation-free AUC is computed only when both later outcomes have at least 20 checkpoint survivors. The raw AUC direction is retained so an apparently strong but directionally inconsistent relationship cannot be called robust. Sample adequacy is checked first. An adequately sampled feature that takes one value at a checkpoint is labelled `constant_feature`: `nested_small_dccs` is zero by construction in the small-to-medium stage, because any small DCC there would resolve the stage.

In [15]:
LIVE_FEATURE_COLUMNS = [
    "target_progress",
    "lower_tmv_live",
    "lower_r_live",
    "upper_tmv_live",
    "upper_r_live",
    "direction_boundaries",
    "outer_agreement_breadth",
    "nested_small_dccs",
    "nested_small_dcc_rate",
]

overlap_records = []
for (case, stage, elapsed_bars), checkpoint in checkpoint_rows.groupby(
    ["case", "stage", "elapsed_bars"],
    sort=True,
):
    target = checkpoint["future_stage_outcome"].eq(AttemptOutcome.PROPAGATED.value).astype(np.int8)
    for feature in LIVE_FEATURE_COLUMNS:
        values = checkpoint[feature].replace([np.inf, -np.inf], np.nan)
        valid = values.notna()
        feature_values = values.loc[valid]
        feature_target = target.loc[valid]
        propagated_count = int(feature_target.sum())
        rejected_count = int(len(feature_target) - propagated_count)

        # Sample adequacy first: a cell with too few of either outcome is insufficient,
        # whether or not its feature happens to take one value.
        if feature_target.nunique() < 2 or min(propagated_count, rejected_count) < MIN_AUC_CLASS_COUNT:
            raw_auc = np.nan
            direction = "insufficient_sample"
            orientation_free_auc = np.nan
        elif feature_values.nunique() < 2:
            raw_auc = np.nan
            direction = "constant_feature"
            orientation_free_auc = np.nan
        else:
            raw_auc = roc_auc_score(feature_target, feature_values)
            direction = "higher" if raw_auc >= 0.5 else "lower"
            orientation_free_auc = max(raw_auc, 1.0 - raw_auc)

        overlap_records.append({
            "case": case,
            "stage": stage,
            "elapsed_bars": elapsed_bars,
            "feature": feature,
            "propagated_survivors": propagated_count,
            "rejected_survivors": rejected_count,
            "raw_auc_for_higher_value": raw_auc,
            "stronger_value_direction": direction,
            "orientation_free_auc": orientation_free_auc,
        })

overlap_summary = pd.DataFrame.from_records(overlap_records)
progress_overlap = overlap_summary.loc[
    overlap_summary["feature"].eq("target_progress")
].copy()
display(progress_overlap.set_index(["case", "stage", "elapsed_bars"]).style.format({
    "propagated_survivors": "{:,}",
    "rejected_survivors": "{:,}",
    "raw_auc_for_higher_value": "{:.3f}",
    "orientation_free_auc": "{:.3f}",
}))

In [16]:
progress_figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[stage.value for stage in STAGE_ORDER],
)
for stage_number, stage in enumerate(STAGE_ORDER):
    row_number = stage_number // 2 + 1
    column_number = stage_number % 2 + 1
    stage_rows = progress_overlap.loc[progress_overlap["stage"].eq(stage.value)]
    for case, group in stage_rows.groupby("case", sort=False):
        progress_figure.add_trace(go.Scatter(
            x=group["elapsed_bars"],
            y=group["orientation_free_auc"],
            mode="lines+markers",
            name=case,
            legendgroup=case,
            showlegend=stage_number == 0,
        ), row=row_number, col=column_number)
    progress_figure.add_hline(
        y=0.5,
        line={"color": "#64748b", "dash": "dot"},
        row=row_number,
        col=column_number,
    )
    progress_figure.update_xaxes(title_text="Elapsed bars", row=row_number, col=column_number)
    progress_figure.update_yaxes(
        title_text="Orientation-free AUC",
        range=[0.48, 1.0],
        row=row_number,
        col=column_number,
    )

progress_figure.update_layout(
    title="Diagnostic target-progress separation across cases",
    template="plotly_white",
    height=850,
    legend={"orientation": "h", "y": 1.16, "x": 0.0},
)
progress_figure.show()

## Audit 5: direction and calendar-period stability

These conditional rates use later-resolved stage outcomes. They test concentration and recurrence; they are not causal gates. Like the Audit 2 ladder, they use every resolved lineage; strict-clean is only for live intervals.

In [17]:
direction_parts = []
period_parts = []
for case_id, result in case_results.items():
    stages = result["resolved_stages"]
    direction_summary = stages.groupby(["stage", "target_mode"], sort=True).agg(
        attempts=("lineage_id", "size"),
        propagation_rate=("outcome", lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean()),
    ).reset_index()
    direction_summary.insert(0, "case", case_id.value)
    direction_parts.append(direction_summary)

    period_summary = stages.groupby(["stage", "sample_third"], sort=True, observed=False).agg(
        attempts=("lineage_id", "size"),
        propagation_rate=("outcome", lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean()),
    ).reset_index()
    period_summary.insert(0, "case", case_id.value)
    period_parts.append(period_summary)

direction_summary = pd.concat(direction_parts, ignore_index=True)
period_summary = pd.concat(period_parts, ignore_index=True)

display(direction_summary.set_index(["case", "stage", "target_mode"]).style.format({
    "attempts": "{:,}",
    "propagation_rate": "{:.2%}",
}))
display(period_summary.set_index(["case", "stage", "sample_third"]).style.format({
    "attempts": "{:,}",
    "propagation_rate": "{:.2%}",
}))

## Audit 6: interpretation matrix

The matrix deliberately separates sample adequacy from qualitative recurrence. For each case and stage it records:

- progress AUC at initiation, 20 bars and 80 bars when adequately sampled;
- whether higher progress associates with later propagation;
- the propagation-rate range across target directions;
- the propagation-rate range across calendar thirds.

Both ranges use only strata with at least `MIN_AUC_CLASS_COUNT` attempts, and are left empty when fewer than two strata qualify.

Missing later-stage values are evidence of limited resolution at that checkpoint, not permission to lower the sample standard.

In [18]:
def overlap_lookup(
    case: str, stage: str, elapsed_bars: int, feature: str
) -> pd.Series | None:
    rows = overlap_summary.loc[
        overlap_summary["case"].eq(case)
        & overlap_summary["stage"].eq(stage)
        & overlap_summary["elapsed_bars"].eq(elapsed_bars)
        & overlap_summary["feature"].eq(feature)
    ]
    if rows.empty:
        return None
    if len(rows) != 1:
        raise AssertionError("Expected at most one overlap row.")
    return rows.iloc[0]


interpretation_records = []
for case_id, result in case_results.items():
    case_name = case_id.value
    for stage in STAGE_ORDER:
        initiation = overlap_lookup(case_name, stage.value, 0, "target_progress")
        at_20 = overlap_lookup(case_name, stage.value, 20, "target_progress")
        at_80 = overlap_lookup(case_name, stage.value, 80, "target_progress")
        direction_rates = direction_summary.loc[
            direction_summary["case"].eq(case_name)
            & direction_summary["stage"].eq(stage.value)
            & direction_summary["attempts"].ge(MIN_AUC_CLASS_COUNT),
            "propagation_rate",
        ]
        period_rates = period_summary.loc[
            period_summary["case"].eq(case_name)
            & period_summary["stage"].eq(stage.value)
            & period_summary["attempts"].ge(MIN_AUC_CLASS_COUNT),
            "propagation_rate",
        ]
        interpretation_records.append({
            "case": case_name,
            "stage": stage.value,
            "auc_at_0": np.nan if initiation is None else initiation["orientation_free_auc"],
            "auc_at_20": np.nan if at_20 is None else at_20["orientation_free_auc"],
            "auc_at_80": np.nan if at_80 is None else at_80["orientation_free_auc"],
            "progress_direction_at_20": (
                "missing" if at_20 is None else at_20["stronger_value_direction"]
            ),
            "direction_rate_range": (
                np.nan if len(direction_rates) < 2 else direction_rates.max() - direction_rates.min()
            ),
            "calendar_rate_range": (
                np.nan if len(period_rates) < 2 else period_rates.max() - period_rates.min()
            ),
            "adequate_outer_clock": state_summary.loc[case_name, "adequate_outer_clock"],
            "adequate_lineage_cohort": state_summary.loc[case_name, "adequate_lineage_cohort"],
        })

interpretation_matrix = pd.DataFrame.from_records(interpretation_records)
display(interpretation_matrix.set_index(["case", "stage"]).style.format({
    "auc_at_0": "{:.3f}",
    "auc_at_20": "{:.3f}",
    "auc_at_80": "{:.3f}",
    "direction_rate_range": "{:.2%}",
    "calendar_rate_range": "{:.2%}",
}))

stage_robustness_summary = interpretation_matrix.groupby("stage", sort=False).agg(
    cases=("case", "size"),
    adequate_20_bar_auc=("auc_at_20", "count"),
    median_auc_at_0=("auc_at_0", "median"),
    median_auc_at_20=("auc_at_20", "median"),
    median_auc_at_80=("auc_at_80", "median"),
    higher_progress_cases_at_20=(
        "progress_direction_at_20",
        lambda values: values.eq("higher").sum(),
    ),
    median_direction_rate_range=("direction_rate_range", "median"),
    median_calendar_rate_range=("calendar_rate_range", "median"),
)
stage_robustness_summary.style.format({
    "cases": "{:,}",
    "adequate_20_bar_auc": "{:,}",
    "median_auc_at_0": "{:.3f}",
    "median_auc_at_20": "{:.3f}",
    "median_auc_at_80": "{:.3f}",
    "higher_progress_cases_at_20": "{:,}",
    "median_direction_rate_range": "{:.2%}",
    "median_calendar_rate_range": "{:.2%}",
})

,cases,adequate_20_bar_auc,median_auc_at_0,median_auc_at_20,median_auc_at_80,higher_progress_cases_at_20,median_direction_rate_range,median_calendar_rate_range
stage,,,,,,,,
small_to_medium,6,6,0.556,0.711,0.717,6,1.23%,5.28%
medium_to_large,6,6,0.529,0.718,0.729,6,3.51%,6.96%
large_to_macro,6,6,0.561,0.692,0.679,6,0.95%,3.80%
macro_to_outer,6,6,0.530,0.658,0.664,6,2.87%,6.50%


## Audit 7: comparison-axis summaries

These views keep the three robustness questions separate. Pooling every case into one aggregate would hide whether disagreement comes from symbol, absolute threshold size or grid geometry.

In [19]:
case_metadata = case_design[["symbol", "grid_family", "comparison_axis"]]
comparison_state = state_summary.join(case_metadata)
display(comparison_state[[
    "symbol",
    "grid_family",
    "comparison_axis",
    "outer_events",
    "strict_clean_lineages",
    "broad_agreement_share",
    "fragmented_share",
    "configuration_entropy_bits",
    "adequate_outer_clock",
    "adequate_lineage_cohort",
]].style.format({
    "outer_events": "{:,}",
    "strict_clean_lineages": "{:,}",
    "broad_agreement_share": "{:.2%}",
    "fragmented_share": "{:.2%}",
    "configuration_entropy_bits": "{:.3f}",
}))

outer_reach = depth_summary.loc[depth_summary["depth"].eq(4), [
    "case", "cohort", "lineages", "cumulative_reach_rate", "lineages_reaching_depth",
]].set_index(["case", "cohort"])
display(outer_reach.style.format({
    "lineages": "{:,}",
    "cumulative_reach_rate": "{:.2%}",
    "lineages_reaching_depth": "{:,}",
}))

,symbol,grid_family,comparison_axis,outer_events,strict_clean_lineages,broad_agreement_share,fragmented_share,configuration_entropy_bits,adequate_outer_clock,adequate_lineage_cohort
case,,,,,,,,,,
GBPUSD ratio-1.5 start-15bp,GBPUSD.r,constant_ratio,absolute_scale,254,"1,220",26.34%,29.18%,4.370,True,True
GBPUSD ratio-1.5 start-20bp,GBPUSD.r,constant_ratio,baseline,146,751,26.39%,29.37%,4.372,True,True
GBPUSD ratio-1.5 start-25bp,GBPUSD.r,constant_ratio,absolute_scale,101,508,26.85%,27.18%,4.313,True,True
EURUSD ratio-1.5 start-20bp,EURUSD.r,constant_ratio,cross_symbol,144,647,24.63%,31.00%,4.423,True,True
USDJPY ratio-1.5 start-20bp,USDJPY.r,constant_ratio,cross_symbol,252,"1,070",32.65%,27.79%,4.139,True,True
GBPUSD linear 20/40/60/80/100bp,GBPUSD.r,linear,grid_sensitivity,146,800,28.47%,26.49%,4.160,True,True


# Five-threshold robustness completion gate

The five-threshold phase is complete only when you can explain:

1. Why every symbol uses one common date interval.
2. Why all six cases were declared before their results.
3. Which comparison changes symbol, absolute scale or grid geometry.
4. Why the constant-ratio ladder is primary even if another case has a higher propagation rate.
5. Why outer-event, lineage-accounting (ambiguous, unresolved, unlinked, same-bar) and strict-lineage counts are reported before interpreting percentages, and why the strict-clean P(reach medium) sits below the all-resolved one while later rates need not.
6. Why all 32 configurations need not have equal occupancy.
7. Whether broad agreement, fragmentation and state entropy recur qualitatively.
8. Whether a linked depth ladder appears in every adequately sampled case.
9. Why exact terminal-depth probabilities are case-specific.
10. Why later checkpoints contain selected survivors.
11. Whether target progress gains diagnostic separation after initiation.
12. Whether the raw direction of that association is consistent across cases.
13. Whether any relationship is concentrated in one target direction.
14. Whether any relationship is concentrated in one calendar third.
15. What changes when absolute thresholds move while their ratios remain fixed.
16. What changes when the grid becomes linear while its outer scale remains similar.
17. Why a grid is part of the model specification rather than neutral preprocessing.
18. Why diagnostic structural robustness is not evidence of trading profitability.
19. Why a future regime classifier must add value beyond directly observed hierarchy geometry.
20. Why all model selection must be training-only and evaluated walk-forward.

After this notebook is interpreted, the five-threshold phases (14–18 in the notes' roadmap) are complete. The next phase is multiscale regime representation: define the smallest causal state vector that preserves mode configuration, agreement breadth, propagation depth, fragmentation, clock ages, live progress and firing density before comparing any classifier.